# Comparing LDPC and CASCADE for LEO CubeSat GMCS CV-QKD: Progress Update


**Summary of what I've done so far:**

- Found an off-the-shelf implementation of the Cascade algorithm in cpp (cascade-cpp)[https://github.com/brunorijsman/cascade-cpp].

- Played around with both the CASCADE algorithm and LDPC algorithm, figuring out how the codebases are structured, how they run experiments, how the algorithms themselves work (and fit into the larger CV-QKD scheme) and what parameters each keeps track of.

- Cascade tracks a lot, LDPC not so much -> Implemented a (somewhat) equivalent parameter tracking system in LDPC (some parameters are not comparable/ do not exist due to the nature of the algorithms). Tried to verify both implementations of algorithms by remaking the graphs from publications (/ Graphs from Ed's December Progress update, for LDPC).

- Attempted to compare the algorithms over each parameter by sweeping SnR and key size, aiming to test their sensitivity and operating ranges.

- Attempted to come up with some kind of model/ feasibility matrix for the LEO CubeSat scenario

**Current state of things:** I can run both algorithms on my hardware, and produce plots of specific (tracked) parameters.

**Summary of main findings:**

INCOMPLETE - Just some notes of things to keep in mind when writing results/eval.

- The graphs show that the algorithms were designed/ are usable in completely different circumstances, they confirm what is expected in theory - LDPC works much better at lower SnR ranges (though this depends on the B-Matrix/Syndrome chosen) and CASCADE only really works at high SnR (low QBER), due to the many round trips that have to be made over the classical channel, for every error. In low SnR, there are so many errors that the number of round trips is much higher than the theoretical minimum defined by Shannon's entropy (how does this releate to mutual information I(A:B)) -> large number of reconciliation bits -> low efficiency -> does not even get close to the 98% reconciliation efficiency target.
- Cascade works (efficiency is reasonable) at QBER of 0.1 and below (corresponds to SnR of 0.75+) - LDPC ,with provided B-Matrices, works best at SnR of 0.xx (with efficiency 1.xx, which is just over the theoretical minimum). 
- Changing the Key rate ...
- Changing the SnR ...
- Changing the LDPC B-Matrix ...
- Changing the CASCADE iterations ...
- Worst case performance ...
- Include some numbers of interesting findings: efficiency, impact of latency(2 min pass duration - at what throughput would it suceed)?, CPU use/Power?

**Summary of what is tracked by Cascade-cpp**


```cpp
using namespace Cascade;

Stats::Stats():
    elapsed_process_time(0.0), // Actual thread CPU time (THREAD_CPUTIME_ID, doesn't include sleep or waiting for I/O time)
    elapsed_real_time(0.0), // Wall clock time - Real time (MONOTONIC - includes sleep time, waiting for I/O time etc.)
    normal_iterations(0), // Always 4 for original CASCADE
    biconf_iterations(0), // Number of BICONF iterations - 2 normal + up to 10 biconf iterations
    start_iteration_messages(0), // normal + biconf iterations - Number of messages sent (Total always 4 for original CASCADE)
    start_iteration_bits(0), // iteration number + seed summed over all iterations - see shuffle seed below.
    ask_parity_messages(0), // Number of round-trips, each message can contain many blocks.
    ask_parity_blocks(0), // number of blocks parity was asked for (total across all round-trips) - number of parity checks asked for.
    ask_parity_bits(0), // Number of bits parity was asked for (total across all round-trips) - 16 (header) + nr_blocks * 80 (parity bits)
    reply_parity_bits(0), // Number of bits parity was replied for (total across all round-trips) - 16 (header) + nr_blocks * 1 (parity bit)
    reconciliation_bits(0), // Total bits sent over classical channel: reconciliation_bits = start_iteration_bits + ask_parity_bits + reply_parity_bits
    efficiency(0.0), // Efficiency of the reconciliation process - reconciliation_bits / (key_size * shannon_efficiency)
    reconciliation_bits_per_key_bit(0.0), // Normalized overhead --- reconciliation_bits / key_size --- Classical communication overhead per key bit.
    infer_parity_blocks(0) // Number of blocks whose correct parity was inferred from parent and sibling parities instead of being asked from Alice.

    // SHUFFLE SEED: Original uses shuffle_seed (32 bits for iteration number + 64 for seed) or full shuffle (32 + 32 * shuffle->get_nr_bits() bits).
{
}
```

**The LDPC tracking implementation**

Developed in a new branch (`feature/stat-tracking`) of the `cvqkd-reconciliation` repository locally - I didn't get a chance to ask if I can push new branch to GitHub.

Results are serialised as **NDJSON** (Newline-Delimited JSON - one JSON object per line per operating point) matched to that of cascade-cpp to simplify cross-algorithm parsing. Statistics are accumulated over N independent frames using **Welford's online algorithm** (Welford, 1962), which computes a numerically stable running mean and variance in a single pass with O(1) memory per tracked metric. [https://en.wikipedia.org/wiki/Algorithms_for_calculating_variance#Welford%27s_online_algorithm] - TL;DR Cascade uses 'Naive one pass' method, both produce equivalent results, but Welford's is more stable numerically. 

Note: Cascade averages over N full-key runs, whereas LDPC averages over N frames, meaning sample counts can match directly but for the same statistical significance, we'd also have to match the key size (each LDPC frame depends on B-Matrix, e.g. 256 bits, whereas CASCADE usually works around 10,000 bits per run) - not sure if it's worth/viable packaging LDPC frames into a bigger frame? e.g. k = 4096 bits = 16 * 256 bit frames or if there is a better way to constrain/match the key size, maybe by applying some post_processing/ analytical approach... -> I decided to divide certain metrics by key_size and turn them into _per_key_bit so that both can operate around their optimal key sizes and still get a fair comparison! Doesn't solve FER but that is mentioned further down in this doc.


`QKD_stats.c` - contains Welford helper functions. It also has a function to add an individual LDPC frames' stats into the aggregated welford counters, and print aggregated counters as NDJSON.

Here's a snippet of the stats I have implemented tracking for.

```c
        ...
        stats->algorithm_name, // Always 'LDPC'
        stats->key_size,
        stats->reconciliations, // Number of frames processed
        stats->snr,
        exec_time, // UTC timestamp of start of run
        stats->actual_bit_errors.mean,
            welford_deviation(&stats->actual_bit_errors),
        stats->actual_bit_error_rate.mean,
            welford_deviation(&stats->actual_bit_error_rate),
        stats->remaining_bit_errors.mean,
            welford_deviation(&stats->remaining_bit_errors),
        stats->remaining_bit_error_rate.mean,
            welford_deviation(&stats->remaining_bit_error_rate),
        stats->remaining_frame_error_rate.mean,
            welford_deviation(&stats->remaining_frame_error_rate),
        stats->elapsed_process_time.mean,
            welford_deviation(&stats->elapsed_process_time),
        stats->elapsed_real_time.mean,
            welford_deviation(&stats->elapsed_real_time),
        stats->normal_iterations.mean, // iterations of min-sum decoder used for each frame
            welford_deviation(&stats->normal_iterations),
        stats->reconciliation_bits.mean, // Syndrome size, usually constant for each B-Matrix
            welford_deviation(&stats->reconciliation_bits),
        stats->reconciliation_bits_per_key_bit.mean,
            welford_deviation(&stats->reconciliation_bits_per_key_bit),
        eff_avg, //Efficiency calculated the same way as CASCADE-CPP --- reconciliation_bits / (key_size * shannon_efficiency)
        eff_dev
    );
```


`ldpc_experiment.c` - runs a single full GMCS CVQKD process (Gaussian + 8d recon + ldpc decoder) with LDPC for a specific (B-Matrix, SNR) combo using `libqkd.c` and tracking relevant statistics using `QKD_stats.c`. The means/STDs are calculated over repeated independent LDPC frames. So if frames_per_point: 10000, then for each (matrix, snr) combo, the binary is invoked once, and that one invocation processes 10000 LDPC frames. (For Cascade, mean and sample SD over repeated independent reconciliation runs)

usage: `./build/ldpc_experiment <snr> <seed> <no. of frames to run> <B-Matrix path> <QKD dimension (defaults to 8)>`

#### Experiment runner

Outside the `cvqkd-reconciliation` repository, in my `qunatum-error-correction-codes-comparison` repository I have a config YAML file and a python `run_sweep.py` with UV package manager, which lets me run multiple combos of (B-matrix, SNR) using `ldpc_experiment.c`

example config:

```yaml
experiment_name: "snr_sweep_0_to_7_5_1000frames_per_point"
seed: 123
# Option A: explicit list
# snr_points: [0.001, 0.01, 0.1, 0.5, 1.0, 2.0]
# Option B: geometric range (start, end, step_factor)
snr:
  start: 0.0001
  end: 7.5
  step_factor: 1.1
frames_per_point: 10000
ldpc:
  binary: "cvqkd-reconciliation/build/ldpc_experiment"
  b_matrices_dir: "cvqkd-reconciliation/data/B_matrices"
  b_matrices: ["256x255_z1.coo", "512x511_z4.coo", "1024x1023_z1.coo"]
  qkd_dimension: 8
output:
  raw_dir: "experiments/data/raw"

```

usage of run_sweep.py: `uv run experiments/scripts/run_sweep.py experiments/config/sweep_snr_ldpc.yaml`

**Summary of experiment runners**

Cascade processes 10,000 bits per run and averages the stats over 10,000 runs, whereas LDPC runs B-Matrix size so (256, 2048 and 1024) bits per frame and averages the stats over 10,000 runs (or frames). A B-Matrix with size 10,000 bits would most closely resemble cascade in terms of FER, clock times and useful bits per pass, but we can scale metrics to per_key_bit to get comparable results. This is why STD of CASCADE is much lower, because we process a lot more bits.



**Verifying the correctness of both algorithms**

The main verification of correctness comes from comparing my graphs to that of widely known and publically accepted literature.

Cascade also has full unit-tests for testing the functionality of each component individually, LDPC has framework but no test implementations. (Might need to make some?)

### Cascade

Efficiency graph created via my workflow, matches the original cascade-cpp authors' graph, which matches that of popular Cascade literature - specifically Figure 1 from "Demystifying the Information Reconciliation Protocol Cascade"

#### My Efficiency Graph

Created with my workflow
Second graph is extended (up to 0.5 QBER, about 0 SNR) vs LDPC.

DOES NOT INCLUDE MODIFIED CASCADE (refer to question at the end about including all CASCADE variations)

![My Efficiency Graph](../graphs/VerificationCascadeOriginal0.png)
![My Efficiency Graph](../graphs/VerificationCascadeOriginalLDPC.png)
<!-- ![My Efficiency Graph Extended](../graphs/VerificationCascadeVs%20LDPC.png) -->

#### Cascade CPP author graph

Original Cascade efficiency (black line) in the QBER 0 - 0.1 matches up with my run (red line).

![cascade-cpp author graph](../graphs/cascadec_cpp_author_fig_1.png)

### Actual Figure 1 from 'Demystifying the IR Protocol Cascade'

![Demystifying the IR Protocol Cascade Figure 1](../graphs/Figure_1_demyst.png)

### LDPC

My 1-2*BER graph matches that of the original author (Ed).

(Why are we using this metric as a measure of performance over just BER?)


![1-2xBER Graph](../graphs/1-2xBER.png)

#### Ed's December Update Graph

![Ed's graph](../graphs/ed_ldpc_1-2xBER.png)


Not sure if matching to some literature is required/needed?

My xxx graph matches that of Figure x from "Some widely known LDPC literature"? I couldn't find some easily

Surprisingly my FER and Ed's FER graphs do not line up. They are complete opposites so I suspect Ed may have plotted 1-FER instead of FER? Even after the change they don't quite line up, which is probably due to Ed packaging the frames into frame sizes of k bits (4096 in the graph) and considering FER to be 1 when any frame has a BER > 0 (my FER/ graph only considers frames of length k == matrix syndrome size, so for 256x255-z1.tsv each frame is 256 bits (1 info + 255 parity bits), FER is set to 1 when there is at least one remaining BER after LDPC min-sum decoding)

#### My FER

![graph](../graphs/1-FER.png)

#### Ed's FER

![graph](../graphs/ed_fer.png)




## Comparison

**Summary of methodology when comparing them**

Since LDPC and CASCADE differ fundamentally in their communication model - one‑way syndrome transmission vs. interactive parity checks - they cannot be compared on strictly identical terms. However, as **information reconciliation** schemes within the same reverse‑reconciliation GMCS CV‑QKD framework, they share a common set of features: reconciliation efficiency η relative to the **Shannon limit** H(QBER) (binary entropy of the bit error rate), residual BER/FER after correction (currently doesn’t compare directly unless key_size matches - decided to turn stats into _per_key_bit, as does Comparing LDPC and CASCADE Muellar et al 2025), classical channel overhead (bits leaked per key bit), computational cost, and latency sensitivity.

For a fair comparison, I treat the **pre‑correction bit error rate** (QBER) as the primary channel parameter that both algorithms must see in common. On the LDPC side, the experiment sweeps **SNR** over the range of interest for GMCS CV‑QKD (Currently 0.0001 to about 7.5 SNR); for each SNR point the full Gaussian channel + 8D multidimensional reconciliation (Leverrier–Grangier) pipeline is applied, and the resulting **actual_bit_error_rate** is measured empirically from the hard decisions on the LLRs. This measured `actual_bit_error_rate` is then taken as the definition of the operating‑point QBER for that SNR — I do **not** rely on the BPSK closed‑form formula to characterise the CV‑QKD channel, because the Gaussian + 8D mapping has a different BER–SNR relationship.

( I've had a quick look into the BPSK formula, QBER ≈ ½ · erfc(√SNR), but since it doesn't match the channel (gaussian + 8d rotation) I don't see it having any use case, it's scattered about in various parts of the project (mainly graphing). Will remove once I confirm it's not needed [https://www.salimwireless.com/2025/11/understanding-q-function-in-bpsk.html] )

On the CASCADE side, I run experiments over **requested_bit_error_rate** (binary symmetric channel) and then **match** those runs to the LDPC operating points by aligning on this measured QBER: for each LDPC SNR point with measured `actual_bit_error_rate`, I either (i) run CASCADE at the same requested bit error rate, or (ii) interpolate within a dense CASCADE QBER sweep to obtain metrics at the closest available QBER. When I plot “metric vs QBER”, the x‑axis is therefore always the **measured pre‑correction BER**: for LDPC this is `actual_bit_error_rate` after the Gaussian + 8D preprocessing; for CASCADE it is the requested / observed bit error rate of its binary symmetric channel.

Both algorithms are run N times per operating point (N frames for LDPC, N full reconciliations for CASCADE). For LDPC, Per-run statistics — mean and **sample** standard deviation — are accumulated online using **Welford's online algorithm** (Welford, 1962), which computes a numerically stable running mean and variance in a single pass with O(1) memory per tracked metric, avoiding the catastrophic cancellation that can arise in a one-pass/two-pass variance calculation at large N. Results are serialised as **NDJSON** (Newline-Delimited JSON), one object per operating point, in a format matched to that of cascade-cpp. CASCADE implementation relies on one-pass variance calculations that result in comparable mean and sample standard deviation statistics.

The LDPC decoder implements **Min-Sum decoding**, a low-complexity approximation of the **Sum-Product algorithm** (Belief Propagation on a Tanner graph). The 8D preprocessing step is the **Leverrier–Grangier multidimensional reconciliation** scheme (Leverrier et al., 2008), which maps Gaussian quadrature values to bits by selecting a unit hypercube corner; the post-rotation values serve as **log-likelihood ratios (LLRs)** for the Min-Sum decoder, where magnitude encodes reliability (soft information) and sign encodes the hard decision. CASCADE variants tested include the **original** (4 normal passes) + potential for others e.g. BICONF, option 7 etc.

CASCADE, by contrast, is an **interactive parity-based information reconciliation protocol** operating over a binary symmetric channel model. Alice and Bob repeatedly partition the key into shuffled blocks, exchange block parities over the authenticated classical channel, and use **binary search / dichotomic search** to localise errors inside blocks whose parities disagree. Later passes reshuffle the key and use previously discovered corrections to trigger further **look-back** corrections in earlier partitions. This means CASCADE does not trade extra local computation for better performance in the same way as LDPC; instead it trades **classical interaction** for error-correction performance. That distinction is especially important for the CubeSat setting, because the total number of parity messages and the number of sequential round trips become part of the feasibility question, not just the raw reconciliation efficiency.

LDPC and CASCADE are therefore always compared at **matched channel conditions** in terms of the actual pre‑correction BER they see, even though one is driven by SNR on a Gaussian CV‑QKD channel and the other by QBER on a binary symmetric channel. In other words, the actual methodology for comparing LDPC and CASCADE is based on **matching their performance at the same measured QBER**, with SNR kept explicitly for LDPC to preserve the CV‑QKD interpretation.

Note:

- STD is much higher for LDPC because of the lower key size (LDPC performance is more variable per frame, CASCADE performance is more consistent per run.)

- Mueller et al. 2025 compare 3 different B-Matrices, 1944, 4000 and 2^16 bits respectively, and they compare against CASCADE at the same key_size as the largest LDPC code (so 2^16 bits). This doesn't work for our scenario unless I vary the keysize, k, for LDPC and then maybe do the same? Currently LDPC codes are very low size and setting CASCADE to a low key_size kills its efficiency. I don't know how Ed packaged the frames into bigger frames of size k either.

### Metrics that are comparable directly

#### Efficiency

`efficiency_avg / efficiency_std` 

Same formula for both: f = leaked_bits / (n · H(QBER)). 

Already normalised.

`Mueller et al. 2025` do  this in Figure 3.


![graph](../graphs/eff_1.png)

0.001 SNR is about 0.487 QBER in the LDPC Pipeline (Gaussian + 8d rotation), What's the target range we should be looking at for LEO CubeSat?

Why is efficiency < 1 for LDPC at very high QBER, very low SNR ?

-> `f = leaked_bits / n * Binary Entropy` Binary entropy is max at 0.5 QBER, == 1 -> So for 255x256 B matrix, syndrome len is fixed at 255 bits, so `f = 255 / 256 * 1 == approx 0.996`

![graph](../graphs/eff_2.png)

![graph](../graphs/eff_3.png)

![graph](../graphs/eff_4.png)

#### Actual Bit Error Rate

`actual_bit_error_rate_avg / _std` 

This measures the channel - It's a sanity check plot that shows that the operating points align.
It should be identical for both

![graph](../graphs/actual_bit_error_rate.png)

#### Reamaining Bit Error Rate

`remaining_bit_error_rate_avg / _std` Already normalised, both should approach zero at their respective operating ranges.
Cascade fully corrects all errors because it is not restricted in terms of latency or timing, mentioned more in the next steps section.

![graph](../graphs/remaining_ber.png)


![graph](../graphs/remaining_ber_2.png)

#### Reconciliation Bits Per Key Bit / Reconciliation Bits

`reconciliation_bits_per_key_bit_avg / _std Already normalised per key bit.` 

`Mueller et al. 2025` do  this in Figure 4 but they use a blind protocol for LDPC, over fixed-size like ours.

LDPC are `255/256`, `1023/1024`, `2044/2048` reconciliation bits per key bit, respectively. (Would I need to model a 120s pass, and find total reconciled bits per pass?)

The algorithm does not send a hash/checksum back to confirm LDPC syndrome is correct?

![graph](../graphs/recon_bits_per_key_bit.png)

![graph](../graphs/recon_bits_per_key_bit_ldpc.png)

`reconciliation_bits_avg` graph for completeness, a little pointless since key_size is 10,000, so they look the same.

![graph](../graphs/recon_bits.png)


###  Metrics that need to be divided by key_size

LDPC processes 256 bits per frame, CASCADE processes 10,000 bits per run.

So normalise Elapsed process and real times to `µs per key bit = elapsed_time / key_size`

The graphs show that LDPC does need more processing/real time, especially at very low SNR/High QBER.

#### Elapsed Process Time

`elapsed_process_time_avg`

![graph](../graphs/process_time_1.png)

![graph](../graphs/process_time_2.png)

#### Elapsed Real Time

`elapsed_real_time_avg`

![graph](../graphs/real_time_1.png)

![graph](../graphs/real_time_2.png)

### Special Case: Frame Error Rate (FER)

Assuming we don't change the key size ... LDPC FER = P(any error in 256 bits). CASCADE FER = P(any error in 10,000 bits).

Mueller et al. handle this through their fFER metric (Equation 5 in the paper), which folds FER into the efficiency:
f_FER = (1 - FER) · f + FER / H(q)

Mention BER and FER about LDPC - We want to keep FER high at low SNR/high QBER so that the decoder does not "successfully" decode random noise, but at the target SNR we want to keep FER low, so that it successfully decodes/converges. Why does this not match up with Ed's December update graph? Surely you want to maximise FER when you have only noise, and minimise FER where you have signal?

![Graph](../graphs/rem_fer_full_range.png)

![Graph](../graphs/rem_fer.png)


### Comparing iterations

Iterations in Cascade refers to count protocol rounds: each round shuffles the key, splits it into blocks, asks for parity, and corrects errors. (They are always set to 4 for ORIGINAL CASCADE). Equivalent to round-trips.

Iterations in LDPC refers to number of Min-Sum decoder iterations (vary based on difficulty of syndrome/ likelihood values)

They measure different things and are not directly comparable.

### Constraints Modelling for LEO CubeSat

The final comparison is not just “which algorithm has better efficiency on a plot?”, but “which algorithm is **actually deployable** within the constraints of a LEO CubeSat CV‑QKD link?”. To answer that, I want to translate the measured experiment statistics into a simple **systems model** with explicit pass-time, bandwidth, latency and compute budgets, then judge each algorithm against those budgets in a feasibility matrix.

As a first-order approximation, I model a single satellite pass as a **constant-SNR operating window** of approximately **2 minutes**. This is obviously a simplification — in reality elevation angle, turbulence, pointing, background light and link margin vary continuously over the pass — but it is a reasonable first step for a progress report because it allows each algorithm to be evaluated at a fixed operating point. In later work, this can be extended to a time-varying SNR profile, where the pass is discretised into short intervals and each interval is assigned its own BER / efficiency / throughput estimate.

The quantum side of the link is assumed to run from a **2.0 MHz laser clock**, which gives the upper bound on how fast raw quantum variables arrive and therefore how much reconciliation work the classical side must keep up with. I am not yet converting this directly into final secret-key throughput, because that also depends on parameter estimation, privacy amplification and exactly how many shared bits are extracted per quantum variable after multidimensional reconciliation. However, the 2.0 MHz figure is still useful as a **pressure test**: any reconciliation method whose compute time or classical messaging overhead cannot keep pace with the incoming data stream is unlikely to be viable onboard, even before the rest of the key-distillation stack is added.

The main constraints I want to model are:

- **Limited pass duration**. A candidate method must reconcile a meaningful amount of data within the ~120 s window. This turns the measured per-frame or per-key timings into an end-to-end throughput question: how many successfully reconciled bits can be delivered before the pass ends?
- **Limited onboard compute**. The CubeSat processor budget is much smaller than a desktop workstation, so `elapsed_process_time` and `elapsed_real_time` should be treated as the starting point for a hardware-scaled estimate, not as final onboard numbers. LDPC may benefit from one-way communication but still fail feasibility if the decoder cost per useful bit is too high on actual hardware.
- **Limited memory**. This still needs tighter numbers, but it matters differently for the two schemes: LDPC needs storage for parity-check matrices, LLRs, syndromes and frame buffers, whereas CASCADE needs larger key buffers, shuffle state, block bookkeeping and parity-tracking structures across multiple passes.
- **Classical channel bandwidth**. The satellite classical link is rate-limited (uplink/downlink bands as provided in the system description), so the tracked metrics `reconciliation_bits` and `reconciliation_bits_per_key_bit` can be converted into transmission-time estimates. 
- **Classical channel latency**. This is especially important for CASCADE, because its cost is not only how many bits are sent, but how many **sequential message exchanges** are required. In the tracked stats this appears through quantities such as `ask_parity_messages`; for LDPC, the currently modelled reverse-reconciliation workflow is effectively **one-way, one classical block per frame** (multidimensional-reconciliation side information + syndrome), so its latency cost is much closer to a single transmission than to a long interactive exchange.
- **Reconciliation quality**. A method is only useful if it achieves an acceptable combination of low residual BER / FER and reasonable efficiency. A protocol that looks good on raw efficiency but fails too often, or still leaves too many residual errors, is not viable. [Not sure how I'd argue where the limits of good/bad quality are]

Using these constraints, I want to estimate an end-to-end **pass budget** for each algorithm at each operating point. A simple starting model is:

`T_total ≈ T_compute + T_classical_tx + T_latency`

where:

- `T_compute` comes from the measured `elapsed_process_time` / `elapsed_real_time`, scaled to the expected onboard processor;
- `T_classical_tx ≈ reconciliation_bits / classical_link_rate`;
- `T_latency ≈ n_messages × propagation_or_round-trip_delay`, where `n_messages` is small for LDPC and potentially large for CASCADE.

This naturally leads to a throughput estimate such as:

`useful reconciled bits per pass ≈ (1 - FER) × key_size × number_of_reconciliations_completed_within_120s`

or, equivalently, a bits-per-second figure that can be compared against the incoming raw data rate implied by the 2.0 MHz source. The exact constants still need to be fixed from the CubeSat system description, but the structure of the model is now clear enough to support a feasibility discussion.

In practical terms, this means the LEO CubeSat feasibility question becomes a checklist rather than a single metric:

- Does the algorithm meet the target operating BER / FER in the expected SNR range?
- Does it achieve acceptable reconciliation efficiency (ideally close to the 98% target, i.e. β ≈ 0.98 or η ≈ 1/0.98)?
- Can the required classical communication fit within the available bandwidth budget?
- Can the interaction pattern tolerate the latency budget of the link?
- Can the onboard processor plausibly keep up with the incoming quantum-variable rate over a 2 minute pass?
- Is the memory footprint reasonable for the platform?

This is the basis for the feasibility matrix below; each algorithm / variant can be marked as viable, borderline or non-viable against each constraint, rather than judged only on a single efficiency curve.

### Feasibility Matrix

should be the final condensed output of the constraints model above. Rather than asking only “which algorithm has the best efficiency?”, the matrix asks “which algorithm is actually compatible with the CubeSat system constraints at the target operating point?”.

something like:

| Algorithm / Variant | Target SNR / QBER reachable? | η / β acceptable? | Residual BER / FER acceptable? | Classical bits within budget? | Message / latency budget acceptable? | Compute budget acceptable? | Memory budget acceptable? | Overall |
|---|---|---|---|---|---|---|---|---|
| LDPC 256x255 z1 |  |  |  |  |  |  |  |  |
| LDPC 512x511 z4 |  |  |  |  |  |  |  |  |
| LDPC 1024x1023 z1 |  |  |  |  |  |  |  |  |
| CASCADE original |  |  |  |  |  |  |  |  |
| CASCADE biconf / option7 / etc. |  |  |  |  |  |  |  |  |

I think the most useful way to fill this is with a simple (`yes / borderline / no`) supported by one or two numbers from the plots. For example:

- `η / β acceptable?` can cite the measured efficiency near the target SNR / QBER.
- `Residual BER / FER acceptable?` can cite `remaining_bit_error_rate` and `remaining_frame_error_rate`.
- `Classical bits within budget?` can cite `reconciliation_bits_per_key_bit` or total `reconciliation_bits`.
- `Message / latency budget acceptable?` can cite one-way LDPC messaging versus `ask_parity_messages` for CASCADE.
- `Compute budget acceptable?` can cite `elapsed_process_time` / `elapsed_real_time` per successful frame or per key bit.

This also gives me a clean way to include algorithm variants without drowning the report in plots: even if I do not show every variant in full detail, I can still summarise whether a given variant is promising, borderline, or clearly unsuitable for the LEO CubeSat scenario.

### Next steps

- add `total successfully reconciled bits per pass` as a metric? will need to model time of frame (something like compute + transmission + latency time), and then see how many frames we can get per 120s pass, multiply that by key_size if frame succeeded otherwise 0 bits.

- Compare the variations of CASCADE? None of the currently available cascade-cpp variants seem to have a reasonable efficiency at the likely low-SNR / high-QBER operating points of interest for LEO CV-QKD. There is, however, a newer **modified / improved Cascade** used in the 2025 LDPC-vs-CASCADE comparison, based on later literature that improves the original protocol. That specific improved variant is not implemented directly in cascade-cpp (I believe option 7 is the closest).

- Show which algorithm variation is viable at what SNR for this LEO scenario?

- Run on real hardware? Compare Memory usage?

- What to do with B-Matrices, which ones to compare?

- Can I commit branch to GitHub?

- More thorough computational / memory comparison?

- Make CASCADE use key_size power of 2, as mentioned in the newer 2025 comparison

- How to deal with latency? FER in CASCADE is P(bit error after reconciliation) in key_size(10000 bits), at the moment because we don't model latency (messages are passed instantly), and we don't restrict the pass duration, the FER always ends up as 0 (reconciled key is fully correct).

- Should I do anything with energy estimates?

- What to do next? Need to keep in mind I only have a few weeks left to write up and submit final report at the end of April.
